## Elección del modelo de clasificación🔬 

En este notebook testeo posibles algoritmos de clasificación una vez obtenidas las caras vectorizadas y con sus respectivos labels de emoción.

Pruebo los distintos modelos...

- Supervisado:
    - KNN
    - SVM

In [11]:
import pandas as pd
import numpy as np

In [12]:
def _load_faces(dir: str) -> pd.DataFrame:    
    angry_faces = pd.read_csv(f"{dir}/angry_faces.csv", header=None).values
    happy_faces = pd.read_csv(f"{dir}/happy_faces.csv", header=None).values
    sad_faces = pd.read_csv(f"{dir}/sad_faces.csv", header=None).values
    surprised_faces = pd.read_csv(f"{dir}/surprise_faces.csv", header=None).values
    neutral_faces = pd.read_csv(f"{dir}/neutral_faces.csv", header=None).values
    disgusted_faces = pd.read_csv(f"{dir}/disgust_faces.csv", header=None).values

    #concat every  face into a single matrix with its corresponding label
    _faces = np.concatenate([
        angry_faces,
        happy_faces,
        sad_faces,
        surprised_faces,    
        neutral_faces,
        disgusted_faces
    ], axis=0)

    #labels for each face
    labels = np.concatenate([
        np.full(angry_faces.shape[0], "angry"),
        np.full(happy_faces.shape[0], "happy"),
        np.full(sad_faces.shape[0], "sad"),
        np.full(surprised_faces.shape[0], "surprised"),
        np.full(neutral_faces.shape[0], "neutral"),
        np.full(disgusted_faces.shape[0], "disgusted")
    ], axis=0)

    faces_df = pd.DataFrame(_faces)

    # if last column is not label, add it
    if faces_df[faces_df.columns[-1]].dtype != object:
        faces_df['label'] = labels
    else:
        faces_df.rename(columns={faces_df.columns[-1]: 'label'}, inplace=True)

    return faces_df

pca_faces = _load_faces("../data/pca_faces")
lbp_faces = _load_faces("../data/lbp_faces")
faces = _load_faces("../data/faces")        

### Preprocess

In [ ]:
# configs
_pca_active = True
_normalize_X = True
data = lbp_faces

In [14]:
X, Y = data.iloc[:, :-1].values, data['label'].values
if _normalize_X:
    X = X / 255.0  # normalize pixel values to [0, 1]

##### Dimensionality Reduction

In [15]:
if _pca_active:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=100)
    X = pca.fit_transform(X)

#### Split Train, Test

In [16]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=30)
X_bal, Y_bal = ros.fit_resample(X, Y)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, 
    Y_bal, 
    test_size=0.2, 
    random_state=30,
    stratify=Y_bal
)
X_train.shape, y_train.shape

((69264, 256), (69264,))

In [17]:
print(pd.Series(y_train).value_counts())

happy       11544
disgust     11544
angry       11544
surprise    11544
sad         11544
neutral     11544
Name: count, dtype: int64


In [18]:
print(pd.Series(y_test).value_counts())

angry       2886
happy       2886
sad         2886
disgust     2886
surprise    2886
neutral     2886
Name: count, dtype: int64


### KNN

In [19]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=6)
knn.fit(X_train, y_train)
knn.score(X_test, y_test)

0.632940632940633

### SVM

In [ ]:
# from sklearn import svm
# clf = svm.SVC(kernel='linear')
# clf.fit(X_train, y_train)
# clf.score(X_test, y_test)